# TP PySpark — Iceberg (cours 8)

Notebook exécutable dans VS Code/Jupyter avec le kernel PySpark.

> Pré-requis: environnement Onyxia démarré avec les variables `KUBERNETES_NAMESPACE`, `AWS_S3_ENDPOINT`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` (et souvent `AWS_SESSION_TOKEN`).


## 1) Initialisation robuste Spark + S3 + Iceberg

Cette section:
- valide les variables d'environnement requises ;
- conserve `LAB_BUCKET_NAME` si déjà défini, sinon l'initialise depuis `KUBERNETES_NAMESPACE` ;
- recrée une `SparkSession` propre avec la configuration S3A + Iceberg ;
- configure le catalogue Hadoop `lab` sur `s3a://<bucket>/warehouse`.


In [ ]:
import os
import pyspark

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col, expr, lit
from pyspark.sql.window import Window

REQUIRED_ENV_VARS = [
    "KUBERNETES_NAMESPACE",
    "AWS_S3_ENDPOINT",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
]

missing_vars = [name for name in REQUIRED_ENV_VARS if not os.environ.get(name)]
if missing_vars:
    missing_str = ", ".join(missing_vars)
    raise EnvironmentError(
        "Variables d'environnement Onyxia manquantes: "
        f"{missing_str}. "
        "Vérifiez la configuration du service et exportez ces variables avant d'exécuter le notebook."
    )

if os.environ.get("LAB_BUCKET_NAME"):
    BUCKET = os.environ["LAB_BUCKET_NAME"]
    print(f"LAB_BUCKET_NAME déjà défini, valeur conservée: {BUCKET}")
else:
    os.environ["LAB_BUCKET_NAME"] = os.environ["KUBERNETES_NAMESPACE"]
    BUCKET = os.environ["LAB_BUCKET_NAME"]
    print(f"LAB_BUCKET_NAME non défini: initialisé depuis KUBERNETES_NAMESPACE -> {BUCKET}")

raw_s3_endpoint = os.environ["AWS_S3_ENDPOINT"].strip()
endpoint_host = raw_s3_endpoint.replace("https://", "").replace("http://", "").rstrip("/")
S3_ENDPOINT_HTTPS = f"https://{endpoint_host}"

ICEBERG_PACKAGE = "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0"
WAREHOUSE_PATH = f"s3a://{BUCKET}/warehouse"

if "spark" in globals() and spark is not None:
    try:
        spark.stop()
        print("SparkSession existante arrêtée pour appliquer une configuration propre.")
    except Exception as stop_error:
        print(f"Avertissement: impossible d'arrêter proprement la session existante: {stop_error}")

spark = (
    SparkSession.builder
    .appName("tp-pyspark-iceberg")
    .master("local[*]")
    # S3A
    .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT_HTTPS)
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.session.token", os.environ.get("AWS_SESSION_TOKEN", ""))
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider",
    )
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
    # Iceberg
    .config("spark.jars.packages", ICEBERG_PACKAGE)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.lab", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lab.type", "hadoop")
    .config("spark.sql.catalog.lab.warehouse", WAREHOUSE_PATH)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("SparkSession initialisée.")


## 2) Vérifications initiales


In [ ]:
print(f"Version Spark: {spark.version}")
print(f"Version PySpark: {pyspark.__version__}")
print(f"Bucket LAB utilisé: {BUCKET}")
print(f"Warehouse Iceberg: {WAREHOUSE_PATH}")


In [ ]:
try:
    spark.sql("SHOW NAMESPACES IN lab").show(truncate=False)
    print("Catalogue `lab` accessible.")
except Exception as catalog_error:
    raise RuntimeError(
        "Le catalogue Iceberg `lab` n'est pas accessible. "
        "Vérifiez la connectivité S3, les credentials AWS et la configuration Spark/Iceberg."
    ) from catalog_error


## 3) Créer/charger `lab.silver.orders`

Objectif: éviter les dépendances implicites au kernel précédent.

Stratégie de bootstrap:
1. si `lab.silver.orders` existe déjà, on la recharge ;
2. sinon si `stg_orders` existe (TP 7 exécuté avant), on crée la table Iceberg depuis ce DataFrame ;
3. sinon on tente de lire `s3a://<bucket>/silver/spark/orders.parquet` puis de créer la table.


In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lab.silver")

table_exists = spark.sql("SHOW TABLES IN lab.silver LIKE 'orders'").count() > 0

if table_exists:
    print("La table lab.silver.orders existe déjà: rechargement.")
elif "stg_orders" in globals() and stg_orders is not None:
    print("Création de lab.silver.orders depuis le DataFrame stg_orders (TP 7).")
    stg_orders.writeTo("lab.silver.orders").create()
else:
    parquet_orders_path = f"s3a://{BUCKET}/silver/spark/orders.parquet"
    print(f"Table absente et stg_orders indisponible. Tentative de lecture: {parquet_orders_path}")
    try:
        stg_orders = spark.read.parquet(parquet_orders_path)
        stg_orders.writeTo("lab.silver.orders").create()
        print("Table Iceberg créée depuis les Parquet du TP précédent.")
    except Exception as bootstrap_error:
        raise RuntimeError(
            "Impossible de créer lab.silver.orders. "
            "Exécutez d'abord le TP 7 pour produire stg_orders ou silver/spark/orders.parquet."
        ) from bootstrap_error

stg_orders = spark.table("lab.silver.orders")
stg_orders.show(5, truncate=False)


In [ ]:
spark.sql(
    """
    SELECT count(*) AS orders,
           sum(quantity) AS quantity
    FROM lab.silver.orders
    """
).show()


## 4) Explorer les metadata Iceberg


In [ ]:
spark.sql(
    """
    SELECT snapshot_id, operation, committed_at
    FROM lab.silver.orders.snapshots
    ORDER BY committed_at
    """
).show(truncate=False)


In [ ]:
spark.sql(
    """
    SELECT file_path,
           file_format,
           record_count,
           file_size_in_bytes
    FROM lab.silver.orders.files
    """
).show(truncate=False)


## 5) Ajouter un snapshot (append contrôlé)


In [ ]:
snapshot_count_before = spark.sql(
    "SELECT count(*) AS c FROM lab.silver.orders.snapshots"
).first()["c"]

if snapshot_count_before < 2:
    new_orders = stg_orders.limit(3).withColumn("order_id", expr("uuid()"))
    new_orders.writeTo("lab.silver.orders").append()
    print("Append exécuté: 3 nouvelles lignes ajoutées.")
else:
    print(
        f"Append déjà effectué (snapshots existants: {snapshot_count_before}). "
        "Aucune nouvelle ligne ajoutée pour garder une ré-exécution stable."
    )

spark.sql(
    """
    SELECT snapshot_id, operation, committed_at
    FROM lab.silver.orders.snapshots
    ORDER BY committed_at
    """
).show(truncate=False)

spark.sql("SELECT count(*) AS total_orders FROM lab.silver.orders").show()


## 6) Time travel


In [ ]:
snapshots = spark.sql(
    """
    SELECT snapshot_id
    FROM lab.silver.orders.snapshots
    ORDER BY committed_at
    """
).collect()

if not snapshots:
    raise RuntimeError("Aucun snapshot trouvé dans lab.silver.orders.")

first_snapshot = snapshots[0]["snapshot_id"]
print(f"Premier snapshot: {first_snapshot}")

spark.sql(
    f"""
    SELECT count(*) AS orders_first_snapshot
    FROM lab.silver.orders
    VERSION AS OF {first_snapshot}
    """
).show()

spark.sql(
    """
    SELECT count(*) AS orders_current
    FROM lab.silver.orders
    """
).show()


## 7) Évolution de schéma


In [ ]:
spark.sql("ALTER TABLE lab.silver.orders ADD COLUMN IF NOT EXISTS source STRING")
spark.sql("DESCRIBE lab.silver.orders").show(truncate=False)


In [ ]:
rows_with_source = spark.sql(
    "SELECT count(*) AS c FROM lab.silver.orders WHERE source = 'spark'"
).first()["c"]

if rows_with_source == 0:
    spark.sql(
        """
        INSERT INTO lab.silver.orders
        SELECT order_id, user_id, ordered_at, quantity, product, 'spark'
        FROM lab.silver.orders
        LIMIT 1
        """
    )
    print("Une ligne avec la nouvelle colonne `source` a été insérée.")
else:
    print("Des lignes avec `source='spark'` existent déjà: insertion ignorée.")

spark.sql(
    """
    SELECT source, count(*) AS n
    FROM lab.silver.orders
    GROUP BY source
    ORDER BY n DESC
    """
).show(truncate=False)


## 8) Comparaison Iceberg vs Parquet


In [ ]:
iceberg_count = spark.table("lab.silver.orders").count()
print(f"Nombre de lignes Iceberg: {iceberg_count}")

parquet_orders_path = f"s3a://{BUCKET}/silver/spark/orders.parquet"
print(f"Chemin Parquet de comparaison: {parquet_orders_path}")

try:
    parquet_orders = spark.read.parquet(parquet_orders_path)
    print(f"Nombre de lignes Parquet: {parquet_orders.count()}")
    print("Schéma Parquet:")
    parquet_orders.printSchema()
except Exception as parquet_error:
    print(
        "Lecture Parquet impossible. Vérifiez que le TP 7 a bien produit "
        f"{parquet_orders_path}. Détail: {parquet_error}"
    )


## 9) Nettoyage


In [ ]:
if "spark" in globals() and spark is not None:
    try:
        spark.stop()
        print("SparkSession arrêtée proprement.")
    except Exception as stop_error:
        print(f"Avertissement: arrêt Spark impossible: {stop_error}")
else:
    print("Aucune SparkSession active à arrêter.")
